In [0]:

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType
)
from datetime import datetime, timezone

AUDIT_TABLE = "ecommerce.gold.pipeline_audit"
PIPELINE_NAME = "ecommerce_lakehouse_pipeline"
TASK_NAME = "silver_transformation"

# Read the job run ID passed by the Databricks job parameter.
try:
    job_run_id = dbutils.widgets.get("job_run_id")
except Exception:
    job_run_id = "manual-or-unavailable"

now = datetime.now(timezone.utc).replace(tzinfo=None)

# Find the most recent RUNNING audit record for this job run.
running_records = (
    spark.table(AUDIT_TABLE)
    .filter(
        (spark.table(AUDIT_TABLE)["job_run_id"] == job_run_id)
        & (spark.table(AUDIT_TABLE)["task_name"] == TASK_NAME)
        & (spark.table(AUDIT_TABLE)["run_status"] == "RUNNING")
    )
    .orderBy("run_started_at", ascending=False)
    .select("execution_id", "run_started_at")
    .limit(1)
    .collect()
)

if running_records:
    execution_id = running_records[0]["execution_id"]
    task_started_at = running_records[0]["run_started_at"]
else:
    execution_id = "execution-id-not-found"
    task_started_at = now

schema = StructType([
    StructField("pipeline_name", StringType(), False),
    StructField("task_name", StringType(), False),
    StructField("run_status", StringType(), False),
    StructField("records_processed", LongType(), True),
    StructField("run_started_at", TimestampType(), False),
    StructField("run_ended_at", TimestampType(), False),
    StructField("details", StringType(), True),
    StructField("job_run_id", StringType(), True),
    StructField("execution_id", StringType(), False)
])

record = [(
    PIPELINE_NAME,
    TASK_NAME,
    "FAILED",
    None,
    task_started_at,
    now,
    "Silver transformation failed. Check the silver_transformation task output for the actual exception.",
    job_run_id,
    execution_id
)]

(
    spark.createDataFrame(record, schema=schema)
    .write.format("delta")
    .mode("append")
    .saveAsTable(AUDIT_TABLE)
)

print(
    f"Silver failure recorded. "
    f"job_run_id={job_run_id}, execution_id={execution_id}"
)
